# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the FAIR² dataset package using the [`mlcroissant`](https://mlcommons.github.io/croissant/python/) library. All dataset elements are referenced via their Croissant schema `@id` fields, ensuring unambiguous access and provenance.

### Dataset Source
The dataset source is published as a Croissant schema at: 

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant pandas matplotlib

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print overall dataset details
md = dataset.metadata  # This is an object, not a dict
print(f"{md.name}: {md.description}")

## 2. Data Overview
We will list all available record sets (tables) and their Croissant `@id` fields, as well as the available fields for each record set. This ensures users know which entities are available for extraction.

For clarity, we enumerate all record sets by their `@id`, then for each record set, list its fields and their corresponding `@id`s.

In [ ]:
# List all record sets by @id
record_sets = dataset.record_sets()

print("Available Record Sets (by @id):")
for rs in record_sets:
    print(f"- {rs['@id']}: {rs.get('name', '[no name]')}")

# Print fields (columns) for each record set
for rs in record_sets:
    print(f"\nFields for Record Set '@id': {rs['@id']}")
    if 'fields' in rs:
        for field in rs['fields']:
            field_id = field.get('@id', '[no field @id]')
            fname = field.get('name', '[no name]')
            dtype = field.get('dataType', '[no dataType]')
            print(f"    - {field_id} (name: {fname}, dtype: {dtype})")
    else:
        print("    [No fields listed]")

## 3. Data Extraction
Load data from each record set into pandas DataFrames using their `@id`.

We demonstrate how to extract each record set to a DataFrame, keyed by its Croissant `@id`. Replace the `record_set_id` and `field_id` variables below with the `@id` strings relevant to your analysis as found in the Data Overview.


In [ ]:
# Gather all record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded record set '{record_set_id}' with shape: {df.shape}")

In [ ]:
# Example: Pick a primary clinical record set for inspection (replace with the actual @id from the overview)
# Here, we attempt to find a record set involving 'clinical', 'CRC', or main patient records, otherwise select the first
record_set_id = None
for rs in record_set_ids:
    if any(x in rs.lower() for x in ['clinical', 'patient', 'crc', 'record', 'main', 'data']):
        record_set_id = rs
        break
if record_set_id is None:
    record_set_id = record_set_ids[0]

print(f"\nInspecting record set: {record_set_id}")
print("Columns (by field @id):")
print(dataframes[record_set_id].columns.tolist())

# Show the first few records
dataframes[record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
In this section, we perform sample processing such as filtering numeric fields, normalization, and group-wise aggregation. All processing uses field `@id` references as columns.

Replace `numeric_field_id` and `group_field_id` with the field `@id`s suitable for your dataset (as printed above).

In [ ]:
# For demonstration, select a numeric field and a group field by their @id.
# You may need to modify these according to your actual dataset fields.
df = dataframes[record_set_id]
numeric_field_id = None
group_field_id = None

# Guess numeric field by dtype or common names
for c in df.columns:
    if any(s in c.lower() for s in ["age", "interval", "years", "count", "number", "tumor_size", "duration"]):
        if pd.api.types.is_numeric_dtype(df[c]):
            numeric_field_id = c
            break
        # Otherwise, try casting to numeric below.

# If none found, try to convert candidate columns
if numeric_field_id is None:
    for c in df.columns:
        try:
            df[c] = pd.to_numeric(df[c])
            if pd.api.types.is_numeric_dtype(df[c]):
                numeric_field_id = c
                break
        except Exception:
            continue

# Guess group field (categorical)
for c in df.columns:
    if any(s in c.lower() for s in ["sex", "gender", "group", "anatomy", "location", "msi", "category", "type"]):
        group_field_id = c
        break

if numeric_field_id is None:
    print("No numeric field automatically detected. Please specify the numeric_field_id explicitly.")
else:
    threshold = df[numeric_field_id].dropna().quantile(0.5)  # Median as threshold
    filtered_df = df[df[numeric_field_id] > threshold]

    print(f"Filtered records with {numeric_field_id} > {threshold} (median): {filtered_df.shape[0]} rows\n")
    print(filtered_df[[numeric_field_id]].head())

    # Normalization
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized '{numeric_field_id}':")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Grouped aggregation (mean)
    if group_field_id is not None and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean of {numeric_field_id} by '{group_field_id}':")
        print(grouped_df.head())
    else:
        print("No suitable group field automatically detected. To group, specify group_field_id explicitly.")

## 5. Visualization
Below are example data visualizations using `matplotlib`, referencing fields by their Croissant `@id`. Adjust field ids as necessary for your exploration.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

# Histogram of numeric field
if numeric_field_id is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(6,4))
    df[numeric_field_id].dropna().hist(bins=20)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

# Boxplot of numeric field grouped by group field
if numeric_field_id is not None and group_field_id is not None and \
   numeric_field_id in df.columns and group_field_id in df.columns:
    plt.figure(figsize=(8,5))
    df.boxplot(column=numeric_field_id, by=group_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.suptitle('')
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()


## 6. Conclusion
In this notebook, we have demonstrated how to load, inspect, and process the FAIR² clinical colorectal cancer dataset using `mlcroissant` via Croissant schema `@id` references. You can now further analyze, visualize, or model these data, referencing any field or entity precisely by its `@id` within your code and workflow.

**Tips:**
- Always use `@id` as your key for field, record set, or column references for robust, schema-anchored data science.
- Consult the Croissant schema or the overview section above when adding fields or integrating with external tools.